In [62]:
import networkx as nx
import random
from sklearn.metrics import root_mean_squared_error, r2_score
import math

In [98]:
fixedProbability = 0.005257 #Probability
Maxie_filepath = "Networks/Maxie_real.txt" #Network
Maxie_seed = ["maxieandreison", "missvanavega"] #Seed

observed =  [1, 370,    10,   1,    2,   1, 0, 1, 0, 0, 0, 0, 0, 0]

In [99]:
def importGraph(filepath):
    print(f"Loading graph from {filepath}...")
    G = nx.DiGraph()
    with open(filepath, "r", encoding="utf-8") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:
                followed, follower = parts[1], parts[0]
                G.add_edge(follower, followed)
    print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    return G

G = importGraph(Maxie_filepath)
node_probabilities = {node: fixedProbability for node in G.nodes()}

Loading graph from Networks/Maxie_real.txt...
Graph loaded with 399322 nodes and 415833 edges.


In [34]:
def ICM(G, seed_nodes, node_probabilities, max_days=14):
    activated_nodes = set(seed_nodes)
    new_active = set(seed_nodes)
    daily_viewers = [len(new_active)]
    daily_retweeters = [len(new_active)]

    for day in range(1, max_days):
        next_active = set()
        viewers_today = set(new_active)

        for node in new_active:
            for neighbor in G.successors(node):
                if neighbor not in activated_nodes:
                    viewers_today.add(neighbor)
                    if random.random() < node_probabilities.get(neighbor, 0):
                        next_active.add(neighbor)

        activated_nodes.update(next_active)
        new_active = next_active

        daily_viewers.append(len(viewers_today))
        daily_retweeters.append(len(next_active))

    return daily_viewers, daily_retweeters, activated_nodes


In [35]:
def getDailyRetweets(G, seed_nodes, node_probabilities, max_days=14, iterations=1000):
    daily_sums = [0] * max_days
    daily_views = [0] * max_days

    for j in range(iterations):
        daily_viewers, daily_retweeters, _ = ICM(G, seed_nodes, node_probabilities, max_days=max_days)
        for day in range(max_days):
            daily_sums[day] += daily_retweeters[day]
            daily_views[day] += daily_viewers[day]

    average_retweets = [round(total / iterations, 2) for total in daily_sums]
    average_viewers = [round(total / iterations, 2) for total in daily_views]

    return average_retweets, average_viewers


In [101]:
iterations = 1000
average_retweets, average_viewers = getDailyRetweets(G, Maxie_seed, node_probabilities, 14, iterations)

print("Retweets Analysis")
predicted = average_retweets
print("===============================================================")
print("On Probability:", fixedProbability, "|", iterations, "iterations")
print(f"Average Retweets: {average_retweets}")
print(f"Observed Data: {observed}")
print("===============================================================")

rmse = root_mean_squared_error(observed, predicted)
r2 = r2_score(observed, predicted)
print(f"RMSE: {rmse:.4f}")
print(f"R-squared: {r2:.4f}", f", Accuracy: {r2*100:.4f}")
print()

Retweets Analysis
On Probability: 0.005257 | 1000 iterations
Average Retweets: [2.0, 287.37, 1.14, 0.04, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Observed Data: [1, 370, 10, 1, 2, 1, 0, 1, 0, 0, 0, 0, 0, 0]
RMSE: 22.2231
R-squared: 0.9453 , Accuracy: 94.5285



In [ ]:
iteration_tests = [1, 10, 100, 1000, 10000]

for i in iteration_tests:
    average_retweets, average_viewers = getDailyRetweets(G, Maxie_seed, node_probabilities, 14, i)

    print("Retweets Analysis")
    predicted = average_retweets
    print("===============================================================")
    print("On Probability:", fixedProbability, "|", i, "iterations")
    print(f"Average Retweets: {average_retweets}")
    print(f"Observed Data: {observed}")
    print("===============================================================")

    rmse = root_mean_squared_error(observed, predicted)
    r2 = r2_score(observed, predicted)
    print(f"RMSE: {rmse:.4f}")
    print(f"R-squared: {r2:.4f}", f", Accuracy: {r2*100:.4f}")
    print()

Retweets Analysis
On Probability: 0.005257 | 1 iterations
Average Retweets: [1.0, 295.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Observed Data: [1, 370, 10, 1, 2, 1, 0, 1, 0, 0, 0, 0, 0, 0]
RMSE: 20.2343
R-squared: 0.9546 , Accuracy: 95.4639

Retweets Analysis
On Probability: 0.005257 | 10 iterations
Average Retweets: [1.0, 292.7, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Observed Data: [1, 370, 10, 1, 2, 1, 0, 1, 0, 0, 0, 0, 0, 0]
RMSE: 20.8400
R-squared: 0.9519 , Accuracy: 95.1883

Retweets Analysis
On Probability: 0.005257 | 100 iterations
Average Retweets: [1.0, 286.57, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Observed Data: [1, 370, 10, 1, 2, 1, 0, 1, 0, 0, 0, 0, 0, 0]
RMSE: 22.4381
R-squared: 0.9442 , Accuracy: 94.4221

Retweets Analysis
On Probability: 0.005257 | 1000 iterations
Average Retweets: [1.0, 287.07, 1.95, 0.02, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Observed Data: [1, 370, 10, 1, 2, 1, 0, 1, 0, 0, 0, 

In [70]:
def importGraph(filepath):
    print(f"Loading graph from {filepath}...")
    G = nx.DiGraph()
    with open(filepath, "r", encoding="utf-8") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:
                followed, follower = parts[1], parts[0]
                G.add_edge(follower, followed)
    print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    return G

G = importGraph(Maxie_filepath)
node_probabilities = {
    node: fixedProbability * G.out_degree(node)
    if G.out_degree(node) > 0 else 0.0
    for node in G.nodes()
}

Loading graph from Networks/Maxie_real.txt...
Graph loaded with 399322 nodes and 415833 edges.
